# Задание 1

In [147]:
import pandas as pd

data = pd.read_csv(
    "school_progress.csv",
    index_col="enrollment_id"
)
print(data.head(5))
print(data.shape)
print(data.dtypes)
print(data.isnull().sum())

               student_id   student_name  grade class_letter    city  \
enrollment_id                                                          
10001                2001    Мария Белов      5            A  Москва   
10002                2001    Мария Белов      5            A  Москва   
10003                2001    Мария Белов      5            A  Москва   
10004                2001    Мария Белов      5            A  Москва   
10005                2002  Мария Макаров      9            A   Псков   

                       course        teacher        track     level  \
enrollment_id                                                         
10001                  Python    Е.В. Орлова  programming     basic   
10002                Биология  Т.К. Лебедева      science  advanced   
10003           Робототехника    И.А. Петров  engineering     basic   
10004                  Физика   М.И. Соколов      science     basic   
10005          Веб-разработка     Д.Р. Белов  programming     basic  

# Задание 2

In [148]:
scores = data["average_score"]
print(scores.mean())
print(scores.median())
print(scores.max())
print(scores.min())
print(scores.isnull().sum())
print(scores.nlargest(5))
print((scores>=80).sum())


74.28333333333333
73.69999999999999
97.6
35.0
22
enrollment_id
10075    97.6
10020    96.6
10193    96.3
10264    96.3
10074    96.2
Name: average_score, dtype: float64
96


# Задание 3

In [149]:
data["progress_calc"] = ((data["lessons_completed"] / data["total_lessons"]) * 100).round(2)

data["progress_num"] = data["progress"].str.replace("%", "", regex=False).astype(float)

data["diff"] = (data["progress_num"] - data["progress_calc"]).abs()

max_difference = data["diff"].max()
print(f"Максимальная разница: {max_difference}")


anomalies = data[data["diff"] > 1]
print("Строки с разницей больше 1%:")
print(anomalies)

Максимальная разница: 0.5
Строки с разницей больше 1%:
Empty DataFrame
Columns: [student_id, student_name, grade, class_letter, city, course, teacher, track, level, start_date, end_date, status, lessons_completed, total_lessons, progress, average_score, hours_spent, attempts, email, clubs, comment, progress_calc, progress_num, diff]
Index: []

[0 rows x 24 columns]


# Задание 4

In [150]:
condition_grade = data["grade"].between(8, 11)
condition_course = data["course"].isin(["Python", "Анализ данных", "Робототехника"])
condition_status = data["status"].isin(["active", "completed"])
condition_average_score = data["average_score"].notna() & (data["average_score"] >= 75)

candidates = data[condition_grade & condition_course & condition_status & condition_average_score]

keep = ["student_name", "grade", "course", "status",
"average_score", "hours_spent"]

candidates = candidates[keep]

candidates = candidates.sort_values(
    by=["average_score", "hours_spent"],
    ascending=[False, False]
)

print(candidates.head(10))

                   student_name  grade         course     status  \
enrollment_id                                                      
10020             Софья Яковлев     11         Python     active   
10138          Виктория Фёдоров     11         Python     active   
10090            Алексей Крылов      8  Анализ данных     active   
10178              Артём Иванов     11         Python  completed   
10194              Кирилл Орлов     10  Анализ данных  completed   
10270            Виктория Орлов      8  Анализ данных     active   
10059              Никита Орлов     10  Анализ данных  completed   
10165           Валерия Новиков      9         Python  completed   
10141            Андрей Фёдоров     11         Python     active   
10225              Алина Павлов      8         Python  completed   

               average_score  hours_spent  
enrollment_id                              
10020                   96.6          9.8  
10138                   95.1         21.8  
10090  

# Задание 5

In [151]:
clean = data.copy()
clean["comment_clean"] = clean["comment"].fillna("нет комментариев")
clean["comment_clean"] = clean["comment_clean"].str.lower()
clean["email_login"] = clean["email"].str.split("@").str[0]
clean["email_domain"] = clean["email"].str.split("@").str[1]

print(clean["comment_clean"].head())
print(clean["email_login"].head())
print(clean["email_domain"].head())

clean["has_robotics_club"] = clean["clubs"].isin(["Робототехника"])

print(clean.head(5))

enrollment_id
10001          иногда опаздывает со сдачей
10002                  отличный прогресс  
10003    нужно внимательнее читать условие
10004                     нет комментариев
10005                     нет комментариев
Name: comment_clean, dtype: str
enrollment_id
10001    student2001
10002    student2001
10003    student2001
10004    student2001
10005    student2002
Name: email_login, dtype: object
enrollment_id
10001    school.example
10002    school.example
10003    school.example
10004    school.example
10005    school.example
Name: email_domain, dtype: object
               student_id   student_name  grade class_letter    city  \
enrollment_id                                                          
10001                2001    Мария Белов      5            A  Москва   
10002                2001    Мария Белов      5            A  Москва   
10003                2001    Мария Белов      5            A  Москва   
10004                2001    Мария Белов      5            

# Задание 6

In [152]:
print(clean["average_score"].isna().sum())
print(clean["clubs"].isna().sum())
print(clean["comment"].isna().sum())
print(clean.duplicated(
    subset=["student_id", "course"],
    keep=False).sum())
print(clean.duplicated(subset=["student_id", "course"], keep="first").sum())
print(len(data) - len(clean))


22
23
26
12
6
0


# Задание 7

In [ ]:
course_report = data.groupby("course").agg(
record = ("student_id", "count"),
students = ("student_id", "nunique"),
mean_score=("average_score", "mean"),
mean_hours=("hours_spent", "mean"),
mean_attempts=("attempts", "mean")
).reset_index()

course_report = course_report.sort_values(by="mean_score", ascending=False)

print("Топ-3 курса по среднему баллу:")
print(course_report.head(3))

Топ-3 курса по среднему баллу:
            course  record  students  mean_score  mean_hours  mean_attempts
2  Английский язык      43        40   75.494872   23.746512       2.790698
0           Python      39        39   75.286486   23.679487       2.717949
6    Робототехника      35        35   74.648387   29.082857       2.914286


# Задание 8

In [ ]:
level_report = data.groupby(["course", "level"]).agg(
records = ("student_id", "count"),
mean_score = ("average_score", "mean"),
mean_hours = ("hours_spent", "mean"),
).reset_index()
level_report_filtered = level_report[level_report["records"]>=5]
print(level_report_filtered.sort_values(by='mean_score', ascending=False).head(5))